In [1]:
import pandas as pd
import numpy as np
from src.common.utils import get_root_directory
from src.common.stats import adf_test, pp_test
import matplotlib.pyplot as plt
import numpy as np
from src.preprocessing.data_loader import DataLoader
from sklearn.metrics import root_mean_squared_error, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import mlflow
import mlflow.sklearn
from darts.models import RandomForest
from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.utils.timeseries_generation import datetime_attribute_timeseries
from darts.metrics import mse

In [709]:
root_dir = get_root_directory()
DL = DataLoader(root_dir)
DL.load_data()
DL.set_time_range(start_date="01/01/2022", end_date="31/12/2023")
train, test = DL.data_split(split_size=0.15)

C:\Users\rajdh\Desktop\ETH_Price_Predition\src\preprocessing\data_loader.py:97: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  end_date = pd.to_datetime(end_date)


In [2]:
def collate_array_elements(arrays):
    if not arrays:
        return []
    return [list(group) for group in zip(*arrays)]

## no_feat

In [562]:
test = test["ETH_D_AvgPrc"]
train, val = DL.data_split(split_type="train_val",split_size=0.15)
train = train["ETH_D_AvgPrc"]
val = val["ETH_D_AvgPrc"]

In [558]:
results = []
n_periods=10
experiment_name = "RF_no_feat_train_validation"
mlflow.set_tracking_uri(f"sqlite:///{root_dir}/mlruns/mlruns.db")
mlflow.set_experiment(experiment_name)
val_gapped = val[:-10]

for lags in range(1,6):
    lag_list = [-(i+1) for i in range(lags)]
    for n_estimators in range(100,401,100):
        train_recursive = TimeSeries.from_series(train)
        predictions = []
        with mlflow.start_run(run_name=f"{lag_list}_{n_estimators}"):
            mlflow.log_params({"lags":lag_list, "n_estimators":n_estimators})
            model = RandomForest(lags=lag_list,n_estimators=n_estimators,criterion="squared_error", random_state=42,output_chunk_length=n_periods)
            for t in range(len(val_gapped)):
                model.fit(train_recursive)
                forecast = model.predict(n=n_periods, series=train_recursive)
                predictions.append(forecast.values().flatten().tolist())
                train_recursive = train_recursive.append_values(val_gapped[t:t+1])
            mse_list = []
            rmse_list = []
            mae_list = []
            mape_list = []
            transposed = collate_array_elements(predictions)
            for i in range(len(transposed)):
                mse = mean_squared_error(val[i:-10+i], transposed[i])
                rmse = root_mean_squared_error(val[i:-10+i], transposed[i])
                mae = mean_absolute_error(val[i:-10+i], transposed[i])
                mape = mean_absolute_percentage_error(val[i:-10+i], transposed[i])
                mse_list.append(mse)
                rmse_list.append(rmse)
                mae_list.append(mae)
                mape_list.append(mape)
            avg_mse = sum(mse_list)/len(mse_list)
            avg_rmse = sum(rmse_list)/len(rmse_list)
            avg_mae = sum(mae_list)/len(mae_list)
            avg_mape = sum(mape_list)/len(mape_list)
            mlflow.log_metrics({"avg_mse": avg_mse, "avg_rmse": avg_rmse, "avg_mae": avg_mae, "avg_mape": avg_mape})
            mlflow.end_run()
        print(f"Completed: {lag_list}_{n_estimators}")

Completed: [-1]_100
Completed: [-1]_200
Completed: [-1]_300
Completed: [-1]_400
Completed: [-1, -2]_100
Completed: [-1, -2]_200
Completed: [-1, -2]_300
Completed: [-1, -2]_400
Completed: [-1, -2, -3]_100
Completed: [-1, -2, -3]_200
Completed: [-1, -2, -3]_300
Completed: [-1, -2, -3]_400
Completed: [-1, -2, -3, -4]_100
Completed: [-1, -2, -3, -4]_200
Completed: [-1, -2, -3, -4]_300
Completed: [-1, -2, -3, -4]_400
Completed: [-1, -2, -3, -4, -5]_100
Completed: [-1, -2, -3, -4, -5]_200
Completed: [-1, -2, -3, -4, -5]_300
Completed: [-1, -2, -3, -4, -5]_400


In [563]:
experiment_name = "RF_no_feat_train_test"
mlflow.set_tracking_uri(f"sqlite:///{root_dir}/mlruns/mlruns.db")
mlflow.set_experiment(experiment_name)
n_periods=10
test_gapped = test[:-10]
train_recursive = TimeSeries.from_series(train)
train_recursive = train_recursive.append_values(val)
lag_list = [-1, -2]
n_estimators = 100
mse_list = []
rmse_list = []
mae_list = []
mape_list = []
predictions = []
with mlflow.start_run(run_name=f"{lag_list}_{n_estimators}"):
    mlflow.log_params({"lags":lag_list, "n_estimators":n_estimators})
    model = RandomForest(lags=lag_list,n_estimators=n_estimators,criterion="squared_error", random_state=42, output_chunk_length=n_periods)
    for t in range(len(test_gapped)):
        model.fit(train_recursive)
        forecast = model.predict(n=n_periods, series=train_recursive)
        predictions.append(forecast.values().flatten().tolist())
        train_recursive = train_recursive.append_values(test_gapped[t:t+1])
    mlflow.sklearn.log_model(model, artifact_path=f"{root_dir}/mlruns/rf_no_feat")
    mse_list = []
    rmse_list = []
    mae_list = []
    mape_list = []
    transposed = collate_array_elements(predictions)
    for i in range(len(transposed)):
        mse = mean_squared_error(test[i:-10+i], transposed[i])
        rmse = root_mean_squared_error(test[i:-10+i], transposed[i])
        mae = mean_absolute_error(test[i:-10+i], transposed[i])
        mape = mean_absolute_percentage_error(test[i:-10+i], transposed[i])
        mse_list.append(mse)
        rmse_list.append(rmse)
        mae_list.append(mae)
        mape_list.append(mape)
        mlflow.log_metrics({"mse_daily":mse_list[i], "rmse_daily":rmse_list[i], "mae_daily":mae_list[i], "mape_daily":mape_list[i]}, step=(i+1))
    avg_mse = sum(mse_list)/len(mse_list)
    avg_rmse = sum(rmse_list)/len(rmse_list)
    avg_mae = sum(mae_list)/len(mae_list)
    avg_mape = sum(mape_list)/len(mape_list)
    mlflow.log_metrics({"avg_mse": avg_mse, "avg_rmse": avg_rmse, "avg_mae": avg_mae, "avg_mape": avg_mape})
    mlflow.end_run()
print(f"Completed: {lag_list}_{n_estimators}")

2025/05/21 13:23:33 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Completed: [-1, -2]_100


## feat

In [90]:
root_dir = get_root_directory()
DL = DataLoader(root_dir)
DL.load_data()

In [91]:
columns = ['ETH_D_AvgPrc','D_CRYPTOBERT_AvgScr_In']
publishers = ['Cointelegraph', 'CoinDesk', 'CoinGape', 'Cointelegraph Deutschland', 'U.Today','The Block', 'Decrypt', 'Cointelegraph Brasil', 'Blockworks','br.cointelegraph.com']
# publishers = 'ALL'
DL.merge_selected_data(selected_data=['sentiment_analysis'], select_publishers=publishers)
DL.set_time_range(start_date="01/01/2022", end_date="31/12/2023")

Pandas Apply:   0%|          | 0/2353 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/2353 [00:00<?, ?it/s]

C:\Users\rajdh\Desktop\ETH_Price_Predition\src\preprocessing\feature_generator.py:98: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.input_data[col_score_In] = (


Pandas Apply:   0%|          | 0/2353 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/2353 [00:00<?, ?it/s]

C:\Users\rajdh\Desktop\ETH_Price_Predition\src\preprocessing\feature_generator.py:98: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.input_data[col_score_In] = (


Pandas Apply:   0%|          | 0/2353 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/2353 [00:00<?, ?it/s]

C:\Users\rajdh\Desktop\ETH_Price_Predition\src\preprocessing\data_loader.py:98: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  end_date = pd.to_datetime(end_date)


In [92]:
DL.select_features(features=columns)
train, test = DL.data_split(split_type="test_train",split_size=0.15)
# train, val = DL.data_split(split_type="train_val",split_size=0.15)

In [93]:
train.columns

Index(['ETH_D_AvgPrc', 'D_CRYPTOBERT_AvgScr_In'], dtype='object')

In [94]:
from sklearn.preprocessing import MinMaxScaler
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

In [40]:
features = [col for col in train.columns if col!='ETH_D_AvgPrc']

In [41]:
features

['D_VADER_AvgScr_In',
 'D_VADER_AvgScr_Ex',
 'D_VADER_Sent_AvgIn',
 'D_VADER_Sent_AvgEx',
 'D_FINBERT_AvgScr_In',
 'D_FINBERT_AvgScr_Ex',
 'D_FINBERT_Sent_AvgIn',
 'D_FINBERT_Sent_AvgEx',
 'D_CRYPTOBERT_AvgScr_In',
 'D_CRYPTOBERT_AvgScr_Ex',
 'D_CRYPTOBERT_Sent_AvgIn',
 'D_CRYPTOBERT_Sent_AvgEx']

In [42]:
results = []
n_periods=10
experiment_name = "RF_sa_feat_train_validation"
mlflow.set_tracking_uri(f"sqlite:///{root_dir}/mlruns/mlruns.db")
mlflow.set_experiment(experiment_name)
# lag_list = [-1,-2,-3,-4,-5] 
n_estimators = 300
for feature in features:
    for lags in range(1,6):
        lag_list = [-(i+1) for i in range(lags)]
        x_train = TimeSeries.from_dataframe(pd.DataFrame(x_scaler.fit_transform(pd.DataFrame(train['ETH_D_AvgPrc']))))
        y_train = TimeSeries.from_dataframe(pd.DataFrame(y_scaler.fit_transform(pd.DataFrame(train[feature]))))
        x_val = TimeSeries.from_dataframe(pd.DataFrame(x_scaler.transform(pd.DataFrame(val['ETH_D_AvgPrc']))))
        y_val = TimeSeries.from_dataframe(pd.DataFrame(y_scaler.transform(pd.DataFrame(val[feature]))))
        # x_test = TimeSeries.from_dataframe(pd.DataFrame(x_scaler.transform(pd.DataFrame(test['ETH_D_AvgPrc']))))
        # y_test = TimeSeries.from_dataframe(pd.DataFrame(y_scaler.transform(pd.DataFrame(test[feature]))))
        # number_features = [f"{i}" for i in range(len(feature))]
        x_train, x_val = x_train.with_columns_renamed("0",'ETH_D_AvgPrc'),x_val.with_columns_renamed("0",'ETH_D_AvgPrc')
        y_train, y_val = y_train.with_columns_renamed("0",feature),y_val.with_columns_renamed("0",feature)
        x_val_gapped, y_val_gapped = x_val[:-10], y_val[:-10]
        # x_test_gapped, y_test_gapped = x_test[:-10], y_test[:-10]
        predictions = []
        with mlflow.start_run(run_name=f"{lag_list}_{n_estimators}_{feature}"):
            mlflow.log_params({"lags":lag_list, "n_estimators":n_estimators, "feature":feature,"publishers":publishers})
            model = RandomForest(lags=[-1,-2],n_estimators=n_estimators,lags_past_covariates=lag_list, criterion="squared_error", output_chunk_length=n_periods, random_state=42)
            for t in range(len(x_val_gapped)):
                model.fit(series=x_train, past_covariates=y_train)
                forecast = model.predict(n=n_periods, series=x_train, past_covariates=y_train)
                predictions.append(forecast.values().flatten().tolist())
                x_train = x_train.concatenate(x_val_gapped[t], ignore_time_axis=True)
                y_train = y_train.concatenate(y_val_gapped[t], ignore_time_axis=True)
            mse_list = []
            rmse_list = []
            mae_list = []
            mape_list = []
            transposed = collate_array_elements(predictions)
            x_val = x_scaler.inverse_transform(pd.DataFrame(x_val.values().flatten().tolist()))
            for i in range(len(transposed)):
                inversed = x_scaler.inverse_transform(pd.DataFrame(transposed[i], columns=['ETH_D_AvgPrc']))
                mse = mean_squared_error(x_val[i:-10+i], inversed)
                rmse = root_mean_squared_error(x_val[i:-10+i], inversed)
                mae = mean_absolute_error(x_val[i:-10+i],inversed)
                mape = mean_absolute_percentage_error(x_val[i:-10+i], inversed)
                mse_list.append(mse)
                rmse_list.append(rmse)
                mae_list.append(mae)
                mape_list.append(mape)
            avg_mse = sum(mse_list)/len(mse_list)
            avg_rmse = sum(rmse_list)/len(rmse_list)
            avg_mae = sum(mae_list)/len(mae_list)
            avg_mape = sum(mape_list)/len(mape_list)
            mlflow.log_metrics({"avg_mse": avg_mse, "avg_rmse": avg_rmse, "avg_mae": avg_mae, "avg_mape": avg_mape})
            mlflow.end_run()
        print(f"Completed: {lag_list}_{n_estimators}_{feature}")

Completed: [-1]_300_D_VADER_AvgScr_In
Completed: [-1, -2]_300_D_VADER_AvgScr_In
Completed: [-1, -2, -3]_300_D_VADER_AvgScr_In
Completed: [-1, -2, -3, -4]_300_D_VADER_AvgScr_In
Completed: [-1, -2, -3, -4, -5]_300_D_VADER_AvgScr_In
Completed: [-1]_300_D_VADER_AvgScr_Ex
Completed: [-1, -2]_300_D_VADER_AvgScr_Ex
Completed: [-1, -2, -3]_300_D_VADER_AvgScr_Ex
Completed: [-1, -2, -3, -4]_300_D_VADER_AvgScr_Ex
Completed: [-1, -2, -3, -4, -5]_300_D_VADER_AvgScr_Ex
Completed: [-1]_300_D_VADER_Sent_AvgIn
Completed: [-1, -2]_300_D_VADER_Sent_AvgIn
Completed: [-1, -2, -3]_300_D_VADER_Sent_AvgIn
Completed: [-1, -2, -3, -4]_300_D_VADER_Sent_AvgIn
Completed: [-1, -2, -3, -4, -5]_300_D_VADER_Sent_AvgIn
Completed: [-1]_300_D_VADER_Sent_AvgEx
Completed: [-1, -2]_300_D_VADER_Sent_AvgEx
Completed: [-1, -2, -3]_300_D_VADER_Sent_AvgEx
Completed: [-1, -2, -3, -4]_300_D_VADER_Sent_AvgEx
Completed: [-1, -2, -3, -4, -5]_300_D_VADER_Sent_AvgEx
Completed: [-1]_300_D_FINBERT_AvgScr_In
Completed: [-1, -2]_300_D_FINB

In [758]:
columns = ['ETH_D_AvgPrc','ETH_ES_GasUsd','ETH_ES_BlkCnt','ETH_YF_Vol','ETH_ES_BlkTm','ETH_OL_ChnVol','ETH_ES_VerCon']
DL.select_features(features=columns)
train, test = DL.data_split(split_type="test_train",split_size=0.15)
train, val = DL.data_split(split_type="train_val",split_size=0.15)

In [67]:
results = []
n_periods=10
experiment_name = "RF_btc_feat_train_validation"
mlflow.set_tracking_uri(f"sqlite:///{root_dir}/mlruns/mlruns.db")
mlflow.set_experiment(experiment_name)
lag_list = [-1,-2,-3] 
n_estimators = 300
features = [col for col in columns if col!='ETH_D_AvgPrc']
for n_estimators in [100,200,400]:
    # lag_list = [-(i+1) for i in range(lags)]
    x_train = TimeSeries.from_dataframe(pd.DataFrame(x_scaler.fit_transform(pd.DataFrame(train['ETH_D_AvgPrc']))))
    y_train = TimeSeries.from_dataframe(pd.DataFrame(y_scaler.fit_transform(pd.DataFrame(train[features]))))
    x_val = TimeSeries.from_dataframe(pd.DataFrame(x_scaler.transform(pd.DataFrame(val['ETH_D_AvgPrc']))))
    y_val = TimeSeries.from_dataframe(pd.DataFrame(y_scaler.transform(pd.DataFrame(val[features]))))
    number_features = [f"{i}" for i in range(len(features))]
    x_train, x_val= x_train.with_columns_renamed("0",'ETH_D_AvgPrc'),x_val.with_columns_renamed("0",'ETH_D_AvgPrc')
    y_train, y_val = y_train.with_columns_renamed(number_features,features),y_val.with_columns_renamed(number_features,features)
    x_val_gapped, y_val_gapped = x_val[:-10], y_val[:-10]
    predictions = []
    with mlflow.start_run(run_name=f"{lag_list}_{n_estimators}"):
        mlflow.log_params({"lags":lag_list, "n_estimators":n_estimators, "feature":features})
        model = RandomForest(lags=[-1,-2],n_estimators=n_estimators,lags_past_covariates=lag_list, criterion="squared_error", output_chunk_length=n_periods, random_state=42)
        for t in range(len(x_val_gapped)):
            model.fit(series=x_train, past_covariates=y_train)
            forecast = model.predict(n=n_periods, series=x_train, past_covariates=y_train)
            predictions.append(forecast.values().flatten().tolist())
            x_train = x_train.concatenate(x_val_gapped[t], ignore_time_axis=True)
            y_train = y_train.concatenate(y_val_gapped[t], ignore_time_axis=True)
        mse_list = []
        rmse_list = []
        mae_list = []
        mape_list = []
        transposed = collate_array_elements(predictions)
        x_val = x_scaler.inverse_transform(pd.DataFrame(x_val.values().flatten().tolist()))
        for i in range(len(transposed)):
            inversed = x_scaler.inverse_transform(pd.DataFrame(transposed[i], columns=['ETH_D_AvgPrc']))
            mse = mean_squared_error(x_val[i:-10+i], inversed)
            rmse = root_mean_squared_error(x_val[i:-10+i], inversed)
            mae = mean_absolute_error(x_val[i:-10+i],inversed)
            mape = mean_absolute_percentage_error(x_val[i:-10+i], inversed)
            mse_list.append(mse)
            rmse_list.append(rmse)
            mae_list.append(mae)
            mape_list.append(mape)
        avg_mse = sum(mse_list)/len(mse_list)
        avg_rmse = sum(rmse_list)/len(rmse_list)
        avg_mae = sum(mae_list)/len(mae_list)
        avg_mape = sum(mape_list)/len(mape_list)
        mlflow.log_metrics({"avg_mse": avg_mse, "avg_rmse": avg_rmse, "avg_mae": avg_mae, "avg_mape": avg_mape})
        mlflow.end_run()
    print(f"Completed: {lag_list}_{n_estimators}")

Completed: [-1, -2, -3]_100
Completed: [-1, -2, -3]_200
Completed: [-1, -2, -3]_400


In [95]:
results = []
n_periods=10
experiment_name = "RF_sa_feat_train_test"
mlflow.set_tracking_uri(f"sqlite:///{root_dir}/mlruns/mlruns.db")
mlflow.set_experiment(experiment_name)
lag_list = [-1,-2,-3] 
n_estimators = 300
features = [col for col in columns if col!='ETH_D_AvgPrc']
x_train = TimeSeries.from_dataframe(pd.DataFrame(x_scaler.fit_transform(pd.DataFrame(train['ETH_D_AvgPrc']))))
y_train = TimeSeries.from_dataframe(pd.DataFrame(y_scaler.fit_transform(pd.DataFrame(train[features]))))
x_test = TimeSeries.from_dataframe(pd.DataFrame(x_scaler.transform(pd.DataFrame(test['ETH_D_AvgPrc']))))
y_test = TimeSeries.from_dataframe(pd.DataFrame(y_scaler.transform(pd.DataFrame(test[features]))))
number_features = [f"{i}" for i in range(len(features))]
x_train, x_test = x_train.with_columns_renamed("0",'ETH_D_AvgPrc'),x_test.with_columns_renamed("0",'ETH_D_AvgPrc')
y_train, y_test = y_train.with_columns_renamed(number_features,features),y_test.with_columns_renamed(number_features,features)
x_test_gapped, y_test_gapped = x_test[:-10], y_test[:-10]
predictions = []
with mlflow.start_run(run_name=f"{lag_list}_{n_estimators}_CRYPTO"):
    mlflow.log_params({"lags":lag_list, "n_estimators":n_estimators, "feature":features, "publishers":publishers})
    model = RandomForest(lags=[-1,-2],n_estimators=n_estimators,lags_past_covariates=lag_list, criterion="squared_error", output_chunk_length=n_periods, random_state=42)
    for t in range(len(x_test_gapped)):
        model.fit(series=x_train, past_covariates=y_train)
        forecast = model.predict(n=n_periods, series=x_train, past_covariates=y_train)
        predictions.append(forecast.values().flatten().tolist())
        x_train = x_train.concatenate(x_test_gapped[t], ignore_time_axis=True)
        y_train = y_train.concatenate(y_test_gapped[t], ignore_time_axis=True)
    mse_list = []
    rmse_list = []
    mae_list = []
    mape_list = []
    transposed = collate_array_elements(predictions)
    x_test = x_scaler.inverse_transform(pd.DataFrame(x_test.values().flatten().tolist()))
    for i in range(len(transposed)):
        inversed = x_scaler.inverse_transform(pd.DataFrame(transposed[i], columns=['ETH_D_AvgPrc']))
        mse = mean_squared_error(x_test[i:-10+i], inversed)
        rmse = root_mean_squared_error(x_test[i:-10+i], inversed)
        mae = mean_absolute_error(x_test[i:-10+i],inversed)
        mape = mean_absolute_percentage_error(x_test[i:-10+i], inversed)
        mse_list.append(mse)
        rmse_list.append(rmse)
        mae_list.append(mae)
        mape_list.append(mape)
        mlflow.log_metrics({"mse_daily":mse_list[i], "rmse_daily":rmse_list[i], "mae_daily":mae_list[i], "mape_daily":mape_list[i]}, step=(i+1))
    avg_mse = sum(mse_list)/len(mse_list)
    avg_rmse = sum(rmse_list)/len(rmse_list)
    avg_mae = sum(mae_list)/len(mae_list)
    avg_mape = sum(mape_list)/len(mape_list)
    mlflow.log_metrics({"avg_mse": avg_mse, "avg_rmse": avg_rmse, "avg_mae": avg_mae, "avg_mape": avg_mape})
    mlflow.end_run()
print(f"Completed: {lag_list}_{n_estimators}")

Completed: [-1, -2, -3]_300


In [ ]:
from darts.explainability.shap_explainer import ShapExplainer
shap_explain = ShapExplainer(model)
results = shap_explain.explain()
shap_explain.summary_plot()
shap_explain.force_plot_from_ts()

C:\Users\rajdh\Desktop\ETH_Price_Predition\.venv\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\rajdh\Desktop\ETH_Price_Predition\.venv\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
PermutationExplainer explainer:   0%|                                                                       | 1/606 [00:00<?, ?it/s]C:\Users\rajdh\Desktop\ETH_Price_Predition\.venv\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\rajdh\Desktop\ETH_Price_Predition\.venv\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\rajdh\Desktop\E